<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/bioassay/bioassay_lean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bioassay: Bayesian workflow

**Short Bayesian course — worked example**

\[
\text{data}
\rightarrow
\text{model}
\rightarrow
\text{prior predictive}
\rightarrow
\text{fit}
\rightarrow
LD50
\rightarrow
\text{posterior predictive}
\]

## 0. Setup

In [ ]:
%pip install -q "pymc==6.3.2" "arviz-plots[matplotlib]==1.3.1" "arviz-stats==1.3.2"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz_plots as azp
import arviz_stats as azs

RANDOM_SEED = 20260923
azp.style.use("arviz-variat")

print("PyMC:", pm.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## 1. Data

In [ ]:
dose = np.array([-0.86, -0.30, -0.05, 0.73])
n = np.array([5, 5, 5, 5])
deaths = np.array([0, 1, 3, 5])

bioassay = pd.DataFrame(
    {
        "dose_log_g_ml": dose,
        "animals": n,
        "deaths": deaths,
    }
)
bioassay["proportion_dead"] = bioassay["deaths"] / bioassay["animals"]
bioassay

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(dose, deaths / n, s=70)
ax.set(
    xlabel="Dose log(g/ml)",
    ylabel="Observed proportion dead",
    ylim=(-0.05, 1.05),
)
plt.show()

## 2. Model

\[
y_j\sim\operatorname{Binomial}(n_j,p_j),
\qquad
\operatorname{logit}(p_j)=\alpha+\beta x_j
\]

\[
\alpha\sim N(0,5),
\qquad
\beta\sim\operatorname{HalfNormal}(5).
\]

In [ ]:
with pm.Model() as model:
    # Data containers make the predictor dimension reusable for prediction.
    dose_data = pm.Data("dose", dose, dims="dose_group")
    n_data = pm.Data("n", n, dims="dose_group")

    alpha = pm.Normal("alpha", mu=0, sigma=5)
    beta = pm.HalfNormal("beta", sigma=5)

    p = pm.Deterministic(
        "p",
        pm.math.sigmoid(alpha + beta * dose_data),
        dims="dose_group",
    )
    mean_deaths = pm.Deterministic(
        "mean_deaths",
        n_data * p,
        dims="dose_group",
    )

    # Scientifically useful derived quantities belong in the model too.
    ld50_log_g_ml = pm.Deterministic(
        "LD50_log_g_ml",
        -alpha / beta,
    )
    ld50_mg_ml = pm.Deterministic(
        "LD50_mg_ml",
        1000 * pm.math.exp(ld50_log_g_ml),
    )

    pm.Binomial(
        "deaths",
        n=n_data,
        p=p,
        observed=deaths,
        dims="dose_group",
    )

## 3. Prior predictive

**Question:** What kinds of observed death counts do these priors make plausible?

In [ ]:
with model:
    prior = pm.sample_prior_predictive(
        draws=1000,
        var_names=["alpha", "beta", "p", "mean_deaths", "deaths"],
        random_seed=RANDOM_SEED,
    )

In [ ]:
azp.plot_ppc_interval(
    prior,
    var_names=["deaths"],
    group="prior_predictive",
    ci_probs=(0.50, 0.90),
    ci_kind="hdi",
    point_estimate="median",
)

## 4. Fit and diagnose

In [ ]:
with model:
    idata = pm.sample(
        draws=1000,
        tune=1500,
        chains=4,
        nuts={"target_accept": 0.90},
        random_seed=RANDOM_SEED,
    )

In [ ]:
print("Divergences:", idata["sample_stats"]["diverging"].sum().item())

azs.summary(
    idata,
    var_names=["alpha", "beta"],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_trace_dist(
    idata,
    var_names=["alpha", "beta"],
)

## 5. LD50

\[
LD50=-\frac{\alpha}{\beta}.
\]

It is stored as a PyMC deterministic quantity.

In [ ]:
azs.summary(
    idata,
    var_names=["LD50_log_g_ml", "LD50_mg_ml"],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_dist(
    idata,
    var_names=["LD50_mg_ml"],
    point_estimate="median",
    ci_prob=0.90,
    ci_kind="hdi",
)

## 6. Posterior predictive

**Question:** Can the fitted model generate death counts like those observed?

In [ ]:
with model:
    pm.sample_posterior_predictive(
        idata,
        var_names=["deaths"],
        extend_inferencedata=True,
        random_seed=RANDOM_SEED,
    )

In [ ]:
azp.plot_ppc_interval(
    idata,
    var_names=["deaths"],
    ci_probs=(0.50, 0.90),
    ci_kind="hdi",
    point_estimate="median",
)

## 7. Dose-response prediction

Change pm.Data to a dense grid and let PyMC create an out-of-sample predictions group.

In [ ]:
dose_grid = np.linspace(-1.0, 1.0, 101)

with model:
    pm.set_data(
        {
            "dose": dose_grid,
            "n": np.full(dose_grid.size, 5),
        }
    )

    pm.sample_posterior_predictive(
        idata,
        var_names=["p", "mean_deaths"],
        predictions=True,
        extend_inferencedata=True,
        random_seed=RANDOM_SEED,
    )

In [ ]:
azp.plot_lm(
    idata,
    x="dose",
    y="mean_deaths",
    y_obs="deaths",
    group="predictions",
    plot_dim="dose_group",
    ci_prob=(0.50, 0.90),
    ci_kind="hdi",
    point_estimate="median",
    smooth=False,
)

## 8. Explore

- Remove the positive-slope constraint.
- Change the priors and rerun the prior predictive check.
- Predict a new group at one chosen dose.

Source: Gelman & Vehtari, *Bayesian Workflow*, §3.5.